# Dilution of Meaning: Multi-Text Experiment

This notebook scales the iterative rewriting experiment to process all text files in the `texts/` directory. It also introduces semantic tracking using embeddings and PCA (Principal Component Analysis) to visualize how the meaning of each text "drifts" across iterations.

## Environment Setup

First, we install and import the necessary libraries. This includes `transformers` for the LLM, `sentence-transformers` for embeddings, and `scikit-learn` for PCA.

In [ ]:
%pip install -q sentence-transformers scikit-learn matplotlib tqdm mlx-lm

In [ ]:
import os
import glob
import torch
import matplotlib.pyplot as plt
import numpy as np
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from tqdm.notebook import tqdm
import time
import nltk
from nltk.tokenize import sent_tokenize
from mlx_lm import load, generate, batch_generate

# nltk.download('punkt_tab')

# Set device
device = "cpu"
if torch.backends.mps.is_available():
    device = torch.device("mps")

if torch.cuda.is_available():
    device = toch.device("cuda")

print(device)

## Model Initialization

We initialize two models:
1.  **Generation Model**: The larger LLM used for rewriting (`google/gemma-3n-E2B-it`).
2.  **Embedding Model**: A specialized model for generating semantic vectors (`all-MiniLM-L6-v2`).

In [ ]:
# Large Language Model for Generation
#gen_model_id = "Qwen/Qwen3-0.7B"
# gen_model_id = "Qwen/Qwen3.5-0.8B" 
# sum_model_id = "Qwen/Qwen3.5-0.8B"
gen_model_id = "mlx-community/Qwen3-0.6B-4bit-DWQ-053125" 
sum_model_id = "mlx-community/Qwen3-0.6B-4bit-DWQ-053125" 

print(f"Loading Generation Model: {gen_model_id}...")
# gen = pipeline("text-generation", model=gen_model_id, device=device)
gen, gen_tokenizer = load(gen_model_id)
print(f"Loading Summarization Model: {gen_model_id}...")
#summarize = pipeline("text-generation", model=sum_model_id, device=device)
summarize, sum_tokenizer = load(sum_model_id)
# Sentence Transformer for Embeddings
print("Loading Embedding Model...")
embed_model = SentenceTransformer('all-mpnet-base-v2', device=device)


## Multi-Text Execution Loop

We discover all `.txt` files in the `texts/` directory and run the rewriting experiment on each. We store the text and its embedding at every step.

In [ ]:

# Ensure you have the sentence tokenizer downloaded
nltk.download('punkt', quiet=True)

# --- CONFIGURATION ---
text_files = glob.glob("./texts/*.txt")
iterations = 100
batch_size = 32

# 1. Read all initial files and tokenize into sentences
print("Reading and splitting texts into sentences...")
sentences_data = []

for file_path in text_files:
    file_name = os.path.basename(file_path)
    with open(file_path, "r", encoding='utf-8') as f:
        text = f.read().strip()
        
    # Split text into sentences
    sents = sent_tokenize(text)
    for idx, s in enumerate(sents):
        # Filter out extremely short or empty sentences
        if len(s.strip()) > 5: 
            sentences_data.append({
                'id': f"{file_name}_sent_{idx}",
                'file': file_name,
                'texts': [s],
                'embeddings': []
            })

current_texts = [item['texts'][0] for item in sentences_data]

# 2. Batch compute initial embeddings
print("Computing initial embeddings...")
orig_embeddings = embed_model.encode(current_texts, batch_size=batch_size)

for idx, emb in enumerate(orig_embeddings):
    sentences_data[idx]['embeddings'].append(emb)

# Clear or create output.txt
with open("output_txt_all.txt", "w", encoding='utf-8') as f:
    f.write("Dilution of Meaning: Sentence-by-Sentence Experiment\n\n")
    for item in sentences_data:
        f.write(f"--- ID: {item['id']} ---\n")
        f.write(f"Original:\n{item['texts'][0]}\n\n")

# 3. Loop over iterations
for i in tqdm(range(1, iterations + 1), desc="Processing Iterations"):
    
    # --- SUMMARIZATION STEP (Adapted for sentences) ---
    summarize_prompts = [
        sum_tokenizer.apply_chat_template(
            [
                {'role': 'system', 'content': 'Distill the provided sentence down to its core logical meaning or subject in exactly one brief sentence.'},
                {'role': 'user', 'content': text}
            ],
            add_generation_prompt=True,
            enable_thinking=False
        )
        for text in current_texts
    ]
    
    
    summaries_output =  batch_generate(summarize,sum_tokenizer, prompts=summarize_prompts).texts
    summaries = [
        out
        for out in summaries_output
    ]


    # --- GENERATION STEP (Adapted for sentences) ---
    generate_prompts = [
        gen_tokenizer.apply_chat_template(
            [
                {'role': 'system', 'content': 'Pretend you are the original author. Expand the following concept into a single, detailed, well-written sentence.'},
                {"role": "user", "content": summary}
            ],
            add_generation_prompt=True,
            enable_thinking=False
        )
        for summary in summaries
    ]

    
    
    #regenerated_output = gen(
        #generate_prompts,
        #max_new_tokens=150, # Reduced for sentence-level
        #return_full_text=False,
        #do_sample=True,
        #temperature=0.7,
        #tokenizer_encode_kwargs={"enable_thinking": False},
        #batch_size=batch_size
    #)
    regenerated_output = batch_generate(gen, gen_tokenizer, prompts = generate_prompts).texts
    
    
    generated_texts = [
        out
        for out in regenerated_output
    ]
    
    

    # --- EMBEDDING STEP ---
    new_embeddings = embed_model.encode(generated_texts, batch_size=batch_size)
    
    # --- RECORD KEEPING ---
    with open("output_txt_all.txt", "a", encoding='utf-8') as f:
        f.write(f"=== ITERATION {i} METRICS: Sum TPS: {sum_tps:.2f} | Gen TPS: {gen_tps:.2f} ===\n\n")
        for idx, item in enumerate(sentences_data):
            item['texts'].append(generated_texts[idx])
            item['embeddings'].append(new_embeddings[idx])
            
            f.write(f"--- ID: {item['id']} | Iteration {i} ---\n")
            f.write(f"Summary: {summaries[idx]}\n")
            f.write(f"Generated: {generated_texts[idx]}\n\n")
    
    current_texts = generated_texts

print("\nText processing complete. Generating visualization...")

# ==========================================
# 4. VISUALIZATION: Trajectories & Clusters
# ==========================================

# Extract embeddings into a 3D NumPy array: (num_sentences, iterations + 1, embedding_dim)
all_embeddings = np.array([item['embeddings'] for item in sentences_data])
num_sentences, num_iters, embed_dim = all_embeddings.shape

# Flatten to fit PCA, then reshape back
flat_embeddings = all_embeddings.reshape(-1, embed_dim)
pca = PCA(n_components=2)
flat_pca = pca.fit_transform(flat_embeddings)
pca_embeddings = flat_pca.reshape(num_sentences, num_iters, 2)

plt.figure(figsize=(14, 10))
ax = plt.gca()

# Setup color map for iterations (e.g., coolwarm or viridis)
cmap = plt.get_cmap('viridis')

# Plot trajectories
for i in range(num_sentences):
    # Extract x and y coordinates for this sentence's trajectory
    x = pca_embeddings[i, :, 0]
    y = pca_embeddings[i, :, 1]
    
    # Create line segments
    points = np.array([x, y]).T.reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    
    # Map colors to the progression of iterations
    norm = plt.Normalize(0, num_iters - 1)
    lc = LineCollection(segments, cmap=cmap, norm=norm, alpha=0.3, linewidths=1.5)
    lc.set_array(np.arange(num_iters - 1))
    ax.add_collection(lc)

    # Mark the start point (Original sentence)
    plt.scatter(x[0], y[0], color='black', s=30, marker='x', zorder=5)
    
    # Mark the end point (Final iteration) to see clustering
    plt.scatter(x[-1], y[-1], color='red', s=50, marker='o', edgecolors='white', zorder=6)

# Create a colorbar to represent iterations
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=0, vmax=iterations))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label('Iteration Number')

# Labels and aesthetics
plt.title('Semantic Drift of Sentences Across Iterations (PCA Projection)', fontsize=16)
plt.xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
plt.ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)

# Custom legend for start/end points
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='x', color='w', label='Start (Original)', markerfacecolor='black', markeredgecolor='black', markersize=8),
    Line2D([0], [0], marker='o', color='w', label='End (Final Iteration)', markerfacecolor='red', markersize=10)
]
plt.legend(handles=legend_elements, loc='best')

plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()

# Save and show
plt.savefig('semantic_drift_clusters.png', dpi=300)
print("Graph saved as 'semantic_drift_clusters.png'.")
plt.show()

## Visualization: Semantic Drift

We use PCA to reduce the embeddings to 2D and plot the trajectories of each file. This shows how the "meaning" of the text moves in semantic space as it is rewritten.

In [ ]:
# Flatten all embeddings to fit PCA
all_embeddings = []
labels = []
for file_name, data in results.items():
    all_embeddings.extend(data['embeddings'])
    for i in range(len(data['embeddings'])):
        labels.append((file_name, i))

all_embeddings = np.array(all_embeddings)
print(all_embeddings.shape)

# Apply PCA
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(all_embeddings)

# Plot
plt.figure(figsize=(12, 8))
colors = plt.cm.rainbow(np.linspace(0, 1, len(results)))

for idx, (file_name, data) in enumerate(results.items()):
    # Get start and end indices in the flattened array
    start_idx = idx * (iterations + 1)
    end_idx = start_idx + (iterations + 1)
    
    coords = embeddings_2d[start_idx:end_idx]
    
    # Plot the line (trajectory)
    plt.plot(coords[:, 0], coords[:, 1], color=colors[idx], alpha=0.5, linestyle='--')
    
    # Plot the points (Original and Iterations)
    for i, (x, y) in enumerate(coords):
        marker = 'o' if i == 0 else 'x'
        marker_size = 100 if i == 0 else 50
        plt.scatter(x, y, color=colors[idx], marker=marker, s=marker_size, label=f"{file_name} (orig)" if i == 0 else "")
        plt.text(x, y, f"{i}", fontsize=9)
        
plt.title("Dilution of Meaning: Semantic Drift across Iterations (PCA 2D)")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()